# Склейка поведенческих видео без потери кадров (ffmpeg)

Блокнот склеивает AVI-клипы каждой сессии в один MP4 (H.264, CRF 23). Он заменяет moviepy из `ConcatenateVideos.ipynb`.

**Чем отличается от старого блокнота:**
- Кадры не теряются. moviepy округляет длительность клипа до 0.01 с и теряет примерно один кадр на 10 клипов.
- Кадр N соответствует строке N в `*_VT_TS.csv`. Файл получает итоговое имя только после сверки числа кадров с числом строк.
- В заголовок записывается реальная частота из таймстемпов (например, 66.02 вместо 30 fps).
- Повторный запуск пропускает готовые файлы. Прерванную склейку можно продолжить.

**Как запустить:**
1. Укажите `root` и `output_folder` в ячейке параметров.
2. Запустите все ячейки по порядку.
3. Посмотрите итог склейки и итог проверки. Статус каждой сессии должен быть `ok` или `skipped_ok`.

84 сессии LT1 (84 ГБ) на 6 параллельных процессах склеивались больше 2 часов. Лучше сначала задать одну короткую сессию в `only_sessions`.

Нужен ffmpeg не старше 5.1. Python-пакеты: numpy, pandas, opencv-python (только для проверки).

In [ ]:
from pathlib import Path
import re
import shutil

# паттерн для идентификатора мышесессии
pattern = re.compile(r"[A-Za-z0-9]{3,6}_[A-Za-z]{1,2}\d{2}_(\d{1,2}[DT])(_\d{1,2}T)?")

# укажи путь к папке с папками мышесессий
root = Path(r"e:\Projects\LT1\BehaviorData\1_Raw")

# укажи путь выходных данных
output_folder = Path(r"e:\Projects\LT1\BehaviorData\2_Combined")

# клеить только эти сессии, например ["LT1_J18_2D"]; None - все найденные
only_sessions = None

# файлы таймстемпов (Bonsai) и их единицы: .NET ticks, 100 нс
TS_GLOB = "*_VT_TS.csv"
TS_TICKS_PER_SEC = 1e7

CRF = 23
PRESET = "medium"
WORKERS = 6  # сколько сессий кодировать одновременно
ENC_THREADS = 4  # потоков x264 на сессию

FF = shutil.which("ffmpeg", path=r"C:\ffmpeg\bin") or shutil.which("ffmpeg")
FP = shutil.which("ffprobe", path=r"C:\ffmpeg\bin") or shutil.which("ffprobe")
assert FF and FP, "ffmpeg не найден"
print(FF)

In [ ]:
# сессия = папка с таймстемпом, имя которой подходит под паттерн
sessions = {}
for ts in sorted(root.rglob(TS_GLOB)):
    name = next((p for p in reversed(ts.parent.parts) if pattern.fullmatch(p)), None)
    if name is None:
        print(f"[WARNING] Идентификатор не найден в пути: {ts}")
    elif only_sessions is None or name in only_sessions:
        sessions[name] = ts

print(f"Найдено сессий: {len(sessions)}")

# клипы без таймстемпов склеить нельзя: не с чем сверить число кадров
no_ts = sorted({a.parent for a in root.rglob("*.avi")} - {t.parent for t in root.rglob(TS_GLOB)})
for folder in no_ts:
    print(f"[WARNING] Есть AVI, нет таймстемпов: {folder}")

missing = set(only_sessions or []) - set(sessions)
if missing:
    print(f"[WARNING] Не найдены: {sorted(missing)}")

In [ ]:
import json
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor
from fractions import Fraction

import numpy as np
import pandas as pd

ENC_RATE = 30
ENC_FRAME_TICKS = 512  # 1/15360 с: такой шаг времени у libx264 + mp4 при 30 fps
NOMINAL_TOLERANCE = 0.5  # 29.99 fps при заголовке 30 - это не ошибка заголовка


def list_clips(folder):
    # числовая сортировка: _2.avi раньше _10.avi
    natural = lambda p: [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", p.name)]
    return sorted(folder.glob("*.avi"), key=natural)


def session_rate(ts_path):
    t = np.loadtxt(ts_path, dtype=np.int64)
    measured = (len(t) - 1) / ((t[-1] - t[0]) / TS_TICKS_PER_SEC)
    if abs(measured - ENC_RATE) < NOMINAL_TOLERANCE:
        return len(t), measured, Fraction(ENC_RATE)
    return len(t), measured, Fraction(round(measured, 2)).limit_denominator(100)


def probe(path):
    r = subprocess.run([FP, "-v", "error", "-count_packets", "-select_streams", "v:0", "-show_entries",
                        "stream=nb_read_packets,avg_frame_rate,duration,width,height",
                        "-show_entries", "format=size", "-of", "json", str(path)], capture_output=True, text=True)
    j = json.loads(r.stdout)
    s = j["streams"][0]
    return dict(frames=int(s["nb_read_packets"]), avg_rate=s["avg_frame_rate"], duration=float(s["duration"]),
                width=s["width"], height=s["height"], size_mb=round(int(j["format"]["size"]) / 1e6, 1))


def run(cmd, log):
    cmd = [str(c) for c in cmd]
    r = subprocess.run(cmd, capture_output=True, text=True, errors="replace")
    log.write(" ".join(cmd) + "\n" + r.stderr + "\n")
    if r.returncode:
        raise RuntimeError(f"ffmpeg exit {r.returncode}, см. {log.name}")


def merge(session, ts):
    row = dict(session=session)
    tmp = output_folder / "_tmp"
    final = output_folder / f"{session}.mp4"
    lst, log_path = tmp / f"{session}.txt", tmp / f"{session}.log"
    enc, part = tmp / f"{session}_enc.mp4", tmp / f"{session}_part.mp4"
    try:
        n_ts, measured, rate = session_rate(ts)
        row.update(ts_rows=n_ts, measured_fps=round(measured, 3), set_fps=str(rate))
        if final.exists():
            p = probe(final)
            if p["frames"] == n_ts and Fraction(p["avg_rate"]) == rate:
                return dict(row, **p, status="skipped_ok")
        quoted = (c.as_posix().replace("'", "'\\''") for c in list_clips(ts.parent))
        lst.write_text("".join(f"file '{q}'\n" for q in quoted), encoding="utf-8")
        with open(log_path, "w", encoding="utf-8") as log:
            # Кодируем всегда с меткой 30 fps: x264 при CRF расходует биты на единицу времени,
            # и с меткой 66 fps тот же CRF дал вдвое меньший файл и худшее качество.
            run([FF, "-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", lst, "-map", "0:v:0", "-an",
                 "-vf", f"setpts=N/({ENC_RATE}*TB)", "-fps_mode", "passthrough", "-c:v", "libx264", "-crf", CRF,
                 "-preset", PRESET, "-pix_fmt", "yuv420p", "-threads", ENC_THREADS,
                 "-video_track_timescale", ENC_RATE * ENC_FRAME_TICKS, enc], log)
            # Настоящую частоту пишем без перекодирования: -r с -fps_mode cfr при дробной частоте терял кадр.
            ticks = rate.denominator * 100
            scale = rate.numerator * 100
            bsf = (f"setts=pts=PTS/{ENC_FRAME_TICKS}*{ticks}:dts=DTS/{ENC_FRAME_TICKS}*{ticks}:duration={ticks}"
                   f":time_base=1/{scale},h264_metadata=tick_rate={rate.numerator * 2}/{rate.denominator}")
            run([FF, "-v", "error", "-y", "-i", enc, "-map", "0:v:0", "-c", "copy", "-bsf:v", bsf,
                 "-video_track_timescale", scale, "-movflags", "+faststart", part], log)
        p = probe(part)
        if p["frames"] != n_ts or Fraction(p["avg_rate"]) != rate:
            return dict(row, **p, status=f"FAILED_check, файл оставлен: {part}")
        os.replace(part, final)
        lst.unlink()
        log_path.unlink()
        return dict(row, **p, status="ok")
    except Exception as e:
        return dict(row, status=f"FAILED: {e}")
    finally:
        enc.unlink(missing_ok=True)

In [ ]:
(output_folder / "_tmp").mkdir(parents=True, exist_ok=True)
rows = []
with ThreadPoolExecutor(WORKERS) as ex:
    for r in ex.map(lambda s: merge(*s), sessions.items()):
        rows.append(r)
        print(r["session"], r["status"], flush=True)

manifest = pd.DataFrame(rows)
manifest.to_csv(output_folder / "merge_manifest.csv", index=False)

# _tmp остаётся только с логами и файлами неудачных сессий
if not any((output_folder / "_tmp").iterdir()):
    (output_folder / "_tmp").rmdir()

bad = manifest[~manifest.status.str.endswith("ok")]
print(f"[OK] склеено {len(manifest)} сессий" if bad.empty else f"[FAILED] {bad.session.tolist()}")
manifest

## Проверка результата

Для каждой сессии ячейка сверяет:
- число кадров в MP4, в клипах и в таймстемпах, как их видит OpenCV (им читает видео DeepLabCut);
- кадры MP4 с сырыми кадрами на нескольких стыках клипов и в конце сессии. PSNR (сходство кадров, в дБ) считается по цвету.

Сдвиг на один кадр снижает PSNR примерно до 28-30 дБ. В LT1 на правильной склейке минимум был 36.8 дБ. Проверка выборочная: она не сверяет все кадры.

In [ ]:
import cv2

MIN_PSNR = 35


def clip_length(path):
    c = cv2.VideoCapture(str(path))
    n = int(c.get(cv2.CAP_PROP_FRAME_COUNT))
    c.release()
    return n


def read_frames(path, wanted):
    c = cv2.VideoCapture(str(path))
    out = {}
    for i in range(max(wanted) + 1):
        ok, f = c.read()
        if not ok:
            break
        if i in wanted:
            out[i] = f
    c.release()
    return out


def verify(session, ts):
    n_ts = sum(1 for line in open(ts) if line.strip())
    clips = list_clips(ts.parent)
    sizes = [clip_length(c) for c in clips]
    starts = np.cumsum([0] + sizes[:-1])
    # стыки в начале, на 10-м и 30-м клипе (там moviepy уже терял кадры), в середине и в конце
    nc = len(clips)
    joints = {k for k in (0, 9, 29, nc // 2, nc - 2) if 0 <= k <= nc - 2}
    wanted = {nc - 1: {sizes[-1] - 2, sizes[-1] - 1}}
    for k in joints:
        wanted.setdefault(k, set()).update({sizes[k] - 2, sizes[k] - 1})
        wanted.setdefault(k + 1, set()).update({0, 1})

    v = cv2.VideoCapture(str(output_folder / f"{session}.mp4"))
    frames, fps = int(v.get(cv2.CAP_PROP_FRAME_COUNT)), v.get(cv2.CAP_PROP_FPS)
    psnr, read_fail = [], 0
    for k, local in sorted(wanted.items()):
        raw = read_frames(clips[k], {i for i in local if i >= 0})
        for i, f in raw.items():
            v.set(cv2.CAP_PROP_POS_FRAMES, int(starts[k]) + i)
            ok, x = v.read()
            if not ok:
                read_fail += 1
                continue
            psnr.append(10 * np.log10(255 ** 2 / max(((x.astype(np.float32) - f) ** 2).mean(), 1e-9)))
    v.release()
    good = frames == n_ts == sum(sizes) and read_fail == 0 and bool(psnr) and min(psnr) >= MIN_PSNR
    return dict(session=session, ts_rows=n_ts, clip_frames=sum(sizes), mp4_frames=frames, mp4_fps=round(fps, 3),
                points=len(psnr), read_fail=read_fail, psnr_min=round(min(psnr), 2) if psnr else None,
                status="ok" if good else "FAILED")


# сессии, которые не склеились, проверять не на чем
done = {s: ts for s, ts in sessions.items() if (output_folder / f"{s}.mp4").exists()}
with ThreadPoolExecutor(WORKERS) as ex:
    check = pd.DataFrame(ex.map(lambda s: verify(*s), done.items()))
check.to_csv(output_folder / "verify.csv", index=False)

bad = check[check.status != "ok"]
print(f"[OK] проверено {len(check)} из {len(sessions)} сессий" if bad.empty and len(check) == len(sessions)
      else f"[FAILED] {bad.session.tolist()}, не склеены: {sorted(set(sessions) - set(done))}")
check